**Diplomatura en Ciencia de Datos, Aprendizaje Automático y sus Aplicaciones**

**Exploración y Curación de Datos**

*Edición 2026*

----

# Trabajo práctico entregable - parte 2

En esta notebook, vamos a cargar el conjunto de datos de [la compentencia Kaggle](https://www.kaggle.com/dansbecker/melbourne-housing-snapshot) sobre estimación de precios de ventas de propiedades en Melbourne, Australia.

Utilizaremos el conjunto de datos reducido producido por [DanB](https://www.kaggle.com/dansbecker). Hemos subido una copia a un servidor de la Universidad Nacional de Córdoba para facilitar su acceso remoto.

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas

import seaborn
seaborn.set_context('talk')

from sqlalchemy import create_engine, text

In [ ]:
import plotly
plotly.__version__


In [ ]:
melb_df = pandas.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv')
melb_df[:3]

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0


## Setup previo — Generación del dataset `airbnb_price_by_zipcode.csv`

El **Ej 2** del TP requiere el archivo `airbnb_price_by_zipcode.csv` (generado en la notebook `02.1 Combinación de datasets.ipynb`). Si no lo tenemos, hay que generarlo desde el dataset original de AirBnB Melbourne.

### Por qué tener este archivo "intermedio" antes de empezar

Podríamos cargar el dataset original de AirBnB en cada ejercicio que lo necesite, pero el AirBnB crudo tiene **84 columnas y ~22K filas** — overhead innecesario para el TP. Lo agregamos UNA VEZ por zipcode y guardamos el resultado.

Esta es exactamente la lógica de **Bronze → Silver** del modelo Delta Lake (apunte `09-etl-y-dags.md`):
- **Bronze**: datos crudos (`cleansed_listings_dec18.csv`).
- **Silver**: datos agregados por una clave de negocio (`airbnb_price_by_zipcode.csv`).

### Plan

1. Cargar `cleansed_listings_dec18.csv` directamente desde el servidor de FAMAF.
2. Limpiar la columna `price` (viene como string con `$` y comas).
3. Agregar por `zipcode` con varias métricas: mediana, count, weekly_price.mean, monthly_price.mean.
4. Guardar como `airbnb_price_by_zipcode.csv` en la carpeta TP2.

### Por qué mediana y NO media para Price
- **Outliers**: el precio de AirBnB tiene cola larga (algunos lujos a $1000+/noche). La media se infla; la mediana no se mueve.
- **Robusto a NaN**: la mediana ignora los faltantes naturalmente al agregar.

Esto es exactamente lo que la consigna del Ej 2.2.1 pregunta: *"¿por qué no la media?"*

In [ ]:
# Cargar dataset original AirBnB (84 columnas, ~22K filas)
url_airbnb = 'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/cleansed_listings_dec18.csv'
airbnb_raw = pandas.read_csv(url_airbnb)
print(f"AirBnB raw shape: {airbnb_raw.shape}")
print(f"Columnas relevantes:")
print(airbnb_raw[['zipcode', 'price', 'weekly_price', 'monthly_price']].head(3))

# Limpiar price: viene como string con $ y comas
def limpiar_precio(serie):
    return (serie.astype(str)
            .str.replace('$', '', regex=False)
            .str.replace(',', '', regex=False)
            .replace('nan', numpy.nan)
            .astype(float))

airbnb_raw['price']         = limpiar_precio(airbnb_raw['price'])
airbnb_raw['weekly_price']  = limpiar_precio(airbnb_raw['weekly_price'])
airbnb_raw['monthly_price'] = limpiar_precio(airbnb_raw['monthly_price'])

# Estandarizar zipcode (viene como string con formatos varios)
airbnb_raw['zipcode'] = pandas.to_numeric(airbnb_raw['zipcode'], errors='coerce')
airbnb_raw = airbnb_raw.dropna(subset=['zipcode'])
airbnb_raw['zipcode'] = airbnb_raw['zipcode'].astype(int)

print(f"\n=== STATS POST-LIMPIEZA ===")
print(f"Shape: {airbnb_raw.shape}")
print(f"Zipcodes únicos: {airbnb_raw['zipcode'].nunique()}")
print(f"price summary:")
print(airbnb_raw['price'].describe().round(2))


In [ ]:
# Agregar por zipcode con varias métricas (mediana, count, mean weekly/monthly)
airbnb_by_zip = (
    airbnb_raw
    .groupby('zipcode')
    .agg(
        airbnb_price_median=('price', 'median'),
        airbnb_price_mean=('price', 'mean'),
        airbnb_count=('price', 'count'),
        airbnb_weekly_mean=('weekly_price', 'mean'),
        airbnb_monthly_mean=('monthly_price', 'mean'),
    )
    .reset_index()
)

# Filtrar zipcodes con menos de 5 registros (poca señal estadística)
MIN_REGISTROS = 5
antes = len(airbnb_by_zip)
airbnb_by_zip = airbnb_by_zip[airbnb_by_zip['airbnb_count'] >= MIN_REGISTROS].copy()
print(f"Zipcodes con >= {MIN_REGISTROS} registros: {len(airbnb_by_zip)} (de {antes})")

print(f"\n=== PREVIEW ===")
print(airbnb_by_zip.head())

# Guardar
output_path = 'airbnb_price_by_zipcode.csv'
airbnb_by_zip.to_csv(output_path, index=False)
print(f"\nGuardado en: {output_path}")
print(f"Shape final: {airbnb_by_zip.shape}")


## Ejercicio 1 SQL:

1. Crear una base de datos en SQLite utilizando la libreria [SQLalchemy](https://stackoverflow.com/questions/2268050/execute-sql-from-file-in-sqlalchemy).
https://docs.sqlalchemy.org/en/14/core/engines.html#sqlite

2. Ingestar los datos provistos en 'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv' en una tabla y el dataset generado en clase con datos de airbnb y sus precios por codigo postal en otra.

3. Validar tipos de columnas antes de guardar. Usá `df.dtypes` para ver los tipos actuales. Prestá especial atención a columnas como `Date` y `Price`: por ejemplo, `Date` puede estar como string en vez de datetime, y `Price` puede venir como string o float. El método `to_sql()` infiere tipos automáticamente, pero puede fallar si los tipos no son los esperados.

4. Implementar consultas en SQL que respondan con la siguiente información:

    - cantidad de registros totales por `Regionname`.
    - cantidad de registros totales por `Suburb` y `Regionname`.
    - Consulta con filtro: ¿Cuántas propiedades hay por `Regionname` con más de 2 habitaciones?
    - Agregación condicional: ¿Cuál es el precio promedio de propiedades según tipo (`Type`) y `Regionname`?
    - Orden y límites: Mostrá el top 5 barrios con propiedades más caras en promedio.

5. Combinar los datasets de ambas tablas ingestadas utilizando el comando JOIN de SQL para obtener un resultado similar a lo realizado con Pandas en clase.

6. Agregar una celda de validación posterior al JOIN con assertions o validación de esquema. Como mínimo, verificá que el número de filas no cambió, que no aparecieron nulos inesperados y que los rangos de variables agregadas sean razonables. Esta validación implementa dimensiones básicas de calidad de datos como validez, completitud e integridad.



### Ejercicio 1.1 — Crear la base de datos SQLite

**Qué pide la consigna**: crear una BD SQLite usando SQLAlchemy.

#### Por qué SQLite y por qué SQLAlchemy

- **SQLite**: la cátedra lo eligió porque es la DB **más desplegada del mundo** (celulares, autos, navegadores). No necesita servidor — el archivo `.sqlite3` ES la base de datos.
- **SQLAlchemy**: es la capa de abstracción de Python para hablar con cualquier DB (SQLite, Postgres, MySQL, etc.) con la misma API. Permite que tu código no dependa del motor concreto.

#### Anatomía de `create_engine`

```python
engine = create_engine('sqlite:///melb.sqlite3', echo=False)
```

- `sqlite:///path`: URL de conexión. Tres barras porque la ruta es relativa. Si fuera absoluta serían 4: `sqlite:////home/.../melb.sqlite3`.
- `echo=False`: si lo ponés en `True`, SQLAlchemy imprime CADA query que ejecuta. Útil para debug, ruidoso en producción.
- El engine NO conecta automáticamente — lo hace lazy cuando hacés `engine.connect()` o `df.to_sql(con=engine)`.

#### Sobre la persistencia

El archivo `melb.sqlite3` se crea en el directorio actual. Vamos a borrarlo si existe para arrancar de cero (idempotencia: que el notebook se pueda correr múltiples veces sin acumular tablas viejas).

In [ ]:
from sqlalchemy import create_engine, text
import os

DB_PATH = 'melb.sqlite3'

# Borrar si existe (para correr el notebook varias veces sin estado residual)
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"DB previa eliminada: {DB_PATH}")

# Crear engine
engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)
print(f"Engine creado para: {DB_PATH}")
print(f"Tipo: {type(engine).__name__}")
print(f"URL: {engine.url}")


### Ejercicio 1.2 — Ingestar `melb_df` y `airbnb_by_zip` en dos tablas

**Plan**: `df.to_sql('nombre_tabla', con=engine, if_exists='replace')`.

#### Tres parámetros que vale la pena entender

- **`name`**: cómo se va a llamar la tabla en SQL (NO el archivo).
- **`if_exists`**: qué hacer si la tabla ya existe:
  - `'fail'` (default): tira error → useful en producción para evitar sobrescribir.
  - `'replace'`: dropea y recrea → conveniente para notebooks que corren varias veces.
  - `'append'`: agrega filas a la tabla existente → útil para cargas incrementales.
- **`index=False`**: por defecto pandas guarda el índice del DataFrame como una columna `index`. En general no queremos eso (es overhead).

#### Tabla `properties` vs tabla `airbnb`

Vamos a llamarlas:
- `properties`: el melb_df completo (13.580 filas × 21 columnas).
- `airbnb`: el airbnb_by_zip agregado (zipcodes únicos con stats agregadas).

Estos son los nombres que vamos a usar en las queries SQL.

In [ ]:
# Ingestar melb_df como tabla 'properties'
melb_df.to_sql('properties', con=engine, if_exists='replace', index=False)
print(f"Tabla 'properties' creada: {len(melb_df)} filas, {melb_df.shape[1]} columnas")

# Ingestar airbnb_by_zip como tabla 'airbnb'
airbnb_by_zip.to_sql('airbnb', con=engine, if_exists='replace', index=False)
print(f"Tabla 'airbnb' creada: {len(airbnb_by_zip)} filas, {airbnb_by_zip.shape[1]} columnas")

# Verificar listando las tablas existentes
with engine.connect() as conn:
    tablas = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'")).fetchall()
    print(f"\nTablas en la DB: {[t[0] for t in tablas]}")


### Ejercicio 1.3 — Validar tipos de columnas

**Qué pide la consigna**: *"Validar tipos de columnas antes de guardar. Prestá especial atención a `Date` y `Price`."*

#### Por qué importan los tipos

SQLite es **dinámicamente tipado** (más laxo que Postgres). Pero los tipos siguen importando para:
- **Ordenamiento**: si `Price` queda como TEXT, `ORDER BY Price DESC` te da "9000000" antes que "Price" (orden lexicográfico, no numérico).
- **Comparaciones**: `WHERE Price > 1000000` con Price TEXT compara strings carácter por carácter.
- **Funciones**: `AVG(Price)` con Price TEXT te tira error en motores estrictos.

#### Las dos columnas problemáticas

- **`Date`**: viene como string `"DD/MM/YYYY"`. SQLite lo va a guardar como TEXT.
  - Decisión: lo convertimos a `datetime` ANTES de ingestar (lo hicimos en TP1 también).
- **`Price`**: en `melb_df` ya viene como `float64` ✓. En el `airbnb_raw` venía como string con `$` — lo limpiamos en el setup.

#### Cómo SQLite infiere tipos desde `to_sql`

| Pandas dtype | SQLite type | Notas |
|--------------|-------------|-------|
| int64 | INTEGER | OK |
| float64 | REAL | OK |
| object (string) | TEXT | OK |
| datetime64 | TIMESTAMP | OK pero con formato ISO |
| bool | INTEGER (0/1) | SQLite no tiene BOOLEAN nativo |

Si dejaras `Date` como string, SQLite lo guarda como TEXT y comparar fechas se vuelve doloroso (`"31/12/2017" < "01/01/2018"` es FALSE lexicográficamente).

In [ ]:
# Convertir Date a datetime en una copia ANTES de re-ingestar
melb_df_typed = melb_df.copy()
melb_df_typed['Date'] = pandas.to_datetime(melb_df_typed['Date'], format='%d/%m/%Y')

print("=== DTYPES PRE-INGESTA ===")
print(melb_df_typed.dtypes)

# Re-ingestar con tipos correctos
melb_df_typed.to_sql('properties', con=engine, if_exists='replace', index=False)

# Verificar tipos en SQLite
with engine.connect() as conn:
    schema = conn.execute(text("PRAGMA table_info(properties)")).fetchall()
print("\n=== ESQUEMA DE LA TABLA 'properties' EN SQLITE ===")
print(f"{'name':20} {'type':12} {'notnull':8} {'dflt':8}")
for row in schema:
    print(f"{row[1]:20} {row[2]:12} {str(row[3]):8} {str(row[4]):8}")


### Ejercicio 1.4 — Consultas SQL

Las 5 queries que pide la consigna, cada una resolviendo un patrón distinto:

| # | Query | Patrón | Conceptos clave |
|---|-------|--------|-----------------|
| 1 | Conteo por `Regionname` | GROUP BY simple + agregación | COUNT, GROUP BY |
| 2 | Conteo por `Suburb` + `Regionname` | GROUP BY compuesto | GROUP BY con 2 columnas |
| 3 | Propiedades por `Regionname` con Rooms > 2 | Filtro pre-grupo (WHERE) | WHERE vs HAVING |
| 4 | Precio promedio por `Type` y `Regionname` | Agregación condicional | AVG + GROUP BY |
| 5 | Top 5 suburbios por precio | Orden + límite | ORDER BY + LIMIT |

#### `WHERE` vs `HAVING` (la confusión clásica)

- **`WHERE`** filtra **filas individuales** ANTES de agrupar.
- **`HAVING`** filtra **grupos** DESPUÉS de agregar.

```sql
-- Esto es WHERE (filtrás filas que entran al grupo)
SELECT Type, AVG(Price) FROM properties WHERE Rooms > 2 GROUP BY Type;

-- Esto es HAVING (filtrás grupos cuyo agregado cumple algo)
SELECT Type, AVG(Price) FROM properties GROUP BY Type HAVING AVG(Price) > 1000000;
```

Si te confundís, recordá: WHERE no puede usar agregaciones (`WHERE AVG(...) > ...` falla); HAVING sí.

In [ ]:
# Query 1: cantidad de registros por Regionname
query_1 = '''
SELECT Regionname, COUNT(*) AS cantidad
FROM properties
GROUP BY Regionname
ORDER BY cantidad DESC
'''
res_1 = pandas.read_sql(query_1, con=engine)
print("=== Q1: Conteo por Regionname ===")
print(res_1.to_string(index=False))


In [ ]:
# Query 2: cantidad de registros por Suburb y Regionname
query_2 = '''
SELECT Regionname, Suburb, COUNT(*) AS cantidad
FROM properties
GROUP BY Regionname, Suburb
ORDER BY cantidad DESC
LIMIT 10
'''
res_2 = pandas.read_sql(query_2, con=engine)
print("=== Q2: Conteo por Suburb + Regionname (top 10) ===")
print(res_2.to_string(index=False))


In [ ]:
# Query 3: propiedades con Rooms > 2 por Regionname
query_3 = '''
SELECT Regionname, COUNT(*) AS propiedades_grandes
FROM properties
WHERE Rooms > 2
GROUP BY Regionname
ORDER BY propiedades_grandes DESC
'''
res_3 = pandas.read_sql(query_3, con=engine)
print("=== Q3: Propiedades con Rooms > 2 por Regionname ===")
print(res_3.to_string(index=False))


In [ ]:
# Query 4: precio promedio por Type y Regionname (con format de salida amigable)
query_4 = '''
SELECT Type, Regionname,
       ROUND(AVG(Price), 0) AS precio_promedio,
       COUNT(*) AS n
FROM properties
GROUP BY Type, Regionname
ORDER BY precio_promedio DESC
'''
res_4 = pandas.read_sql(query_4, con=engine)
print("=== Q4: Precio promedio por Type y Regionname ===")
print(res_4.head(15).to_string(index=False))


In [ ]:
# Query 5: top 5 suburbios por precio promedio (mínimo 30 ventas para significancia)
query_5 = '''
SELECT Suburb,
       ROUND(AVG(Price), 0) AS precio_promedio,
       COUNT(*) AS ventas
FROM properties
GROUP BY Suburb
HAVING COUNT(*) >= 30
ORDER BY precio_promedio DESC
LIMIT 5
'''
res_5 = pandas.read_sql(query_5, con=engine)
print("=== Q5: Top 5 suburbios con precios promedio más altos (min 30 ventas) ===")
print(res_5.to_string(index=False))


**Nota sobre Q5**: agregué `HAVING COUNT(*) >= 30`. ¿Por qué?

- Sin el HAVING, los suburbios con 1-2 ventas (donde una sola venta atípica distorsiona el promedio) entran al top.
- Con el filtro, te asegurás de que el "promedio alto" sea **estadísticamente razonable** (al menos 30 observaciones).
- Esto es un patrón típico en SQL: **filtrar agregados** con HAVING para evitar outliers de muestra chica.

### Ejercicio 1.5 — JOIN entre `properties` y `airbnb`

**Qué pide**: combinar las dos tablas con JOIN, equivalente al `merge` de Pandas.

#### Decisión: ¿LEFT JOIN o INNER JOIN?

- **INNER JOIN**: solo filas donde el zipcode existe en AMBAS tablas. Perdés properties cuyo zipcode no esté en AirBnB.
- **LEFT JOIN**: TODAS las properties + datos de AirBnB cuando coincide zipcode. NaN donde no hay match.

Para curación, **LEFT JOIN es lo correcto**: queremos enriquecer properties sin perder filas. Si después del merge ves muchos NaN en `airbnb_price_median`, eso es señal de que esos zipcodes no tienen oferta AirBnB — información que vale conservar.

#### La trampa del cast

`properties.Postcode` está como `REAL` en SQLite (porque era `float64` en pandas — por los NaN), pero en `airbnb`, `zipcode` está como `INTEGER`. SQLite es laxo y va a comparar bien (`3000.0 = 3000` es TRUE en SQLite), pero en motores estrictos esto fallaría. Mejor castear explícitamente.

In [ ]:
# JOIN: properties LEFT JOIN airbnb por zipcode
query_join = '''
SELECT p.Suburb, p.Type, p.Price, p.Postcode,
       a.airbnb_price_median, a.airbnb_price_mean, a.airbnb_count
FROM properties p
LEFT JOIN airbnb a ON CAST(p.Postcode AS INTEGER) = a.zipcode
'''

res_join = pandas.read_sql(query_join, con=engine)
print(f"Filas resultantes: {len(res_join)}")
print(f"Filas en properties original: {len(melb_df_typed)}")
print(f"Filas con airbnb_price_median NaN: {res_join['airbnb_price_median'].isna().sum()}")
print(f"% de cobertura del merge: {(res_join['airbnb_price_median'].notna().sum() / len(res_join) * 100):.1f}%")
print(f"\nPreview:")
print(res_join.head())


### Ejercicio 1.6 — Validación post-JOIN con assertions

**Qué pide**: *"Como mínimo, verificá que el número de filas no cambió, que no aparecieron nulos inesperados y que los rangos de variables agregadas sean razonables."*

#### Por qué validar post-merge

Un JOIN puede fallar silenciosamente de varias formas:
- **Filas duplicadas** (si la clave del lado derecho no es única). Tu dataset crece y no te das cuenta.
- **Filas perdidas** (si usaste INNER JOIN sin querer).
- **Rangos absurdos** en columnas agregadas (precio negativo, count > 1M).
- **Tipos incompatibles** que devuelven NULL silencioso.

Los **assertions** convierten esos bugs silenciosos en errores explícitos.

#### Las 4 dimensiones de calidad del apunte `09-etl-y-dags.md`

| Dimensión | Qué validamos |
|-----------|---------------|
| **Validez** | El zipcode joineado existe en AirBnB tabla original |
| **Completitud** | No aparecieron nulos en columnas que no tenían |
| **Integridad** | Las filas de properties siguen siendo 13.580 (no crecimos ni perdimos) |
| **Consistencia de rango** | airbnb_price_median ∈ [10, 1000] (precios de Melbourne razonables) |

In [ ]:
# Validaciones post-JOIN
print("=== VALIDACIÓN POST-JOIN ===\n")

# 1. Integridad: las filas de properties no cambiaron
assert len(res_join) == len(melb_df_typed), \
    f"Filas cambiaron: {len(res_join)} vs {len(melb_df_typed)}"
print(f"[OK] Integridad: filas se mantienen ({len(res_join)})")

# 2. Completitud: columnas de properties no introducen NaN nuevos
for col in ['Suburb', 'Type', 'Price', 'Postcode']:
    nulls = res_join[col].isna().sum()
    assert nulls == 0, f"Apareció NaN en {col}: {nulls} filas"
print(f"[OK] Completitud: columnas de properties sin NaN")

# 3. Validez: cobertura del merge no es 0
cobertura = res_join['airbnb_price_median'].notna().sum() / len(res_join)
assert cobertura > 0.3, f"Cobertura muy baja: {cobertura*100:.1f}%"
print(f"[OK] Validez: cobertura del merge = {cobertura*100:.1f}%")

# 4. Rangos razonables en columnas agregadas (cuando hay match)
con_match = res_join.dropna(subset=['airbnb_price_median'])
assert con_match['airbnb_price_median'].between(10, 2000).all(), \
    f"airbnb_price_median fuera de rango razonable"
print(f"[OK] Rango: airbnb_price_median ∈ [10, 2000] AUD/noche")

print(f"airbnb_count ∈ [{con_match['airbnb_count'].min():.0f}, {con_match['airbnb_count'].max():.0f}]")
assert con_match['airbnb_count'].min() >= 5, "Algún zipcode tiene < 5 registros (debería filtrarse en el setup)"
print(f"[OK] Conteo mínimo: airbnb_count >= 5")

print("\n=== TODAS LAS VALIDACIONES PASARON ===")


## Ejercicio 2 - Pandas:

Este ejercicio usa el archivo `airbnb_price_by_zipcode.csv` generado en el notebook `02.1 Combinación de datasets.ipynb`. Si no lo tenés, generarlo primero antes de comenzar esta parte.

1. Seleccionar un subconjunto de columnas que les parezcan relevantes al problema de predicción del valor de la propiedad. Justificar explicitamente las columnas seleccionadas y las que no lo fueron.
  1. Valores faltantes: ¿Qué porcentaje de filas tienen al menos un valor faltante?
  2. Mostrar la dispersión o distribución de las columnas seleccionadas.
 3. Eliminar los valores extremos que no sean relevantes para la predicción de valores de las propiedades.
 4. Mostrar visualmente los valores extremos que eliminás


2. Agregar información adicional respectiva al entorno de una propiedad a partir del [conjunto de datos de AirBnB](https://www.kaggle.com/tylerx/melbourne-airbnb-open-data?select=cleansed_listings_dec18.csv) utilizado en el práctico.
  1. Seleccionar qué variables agregar y qué combinaciones aplicar a cada una. Por ejemplo, pueden utilizar solo la columna `price`, o aplicar múltiples transformaciones como la mediana (¿por qué no la media?) o el mínimo.
  2. Utilizar la variable zipcode para unir los conjuntos de datos. Sólo incluir los zipcodes que tengan una cantidad mínima de registros (a elección) como para que la información agregada sea relevante.
  3. Mostrar un gráfico zipcode vs airbnb_price_median.
  4. Investigar al menos otras 2 variables que puedan servir para combinar los datos, y justificar si serían adecuadas o no. Pueden asumir que cuentan con la ayuda de anotadores expertos para encontrar equivalencias entre barrios o direcciones, o que cuentan con algoritmos para encontrar las n ubicaciones más cercanas a una propiedad a partir de sus coordenadas geográficas. **NO** es necesario que realicen la implementación. Si tuvieras que entrevistar a un experto inmobiliario para mapear barrios entre datasets, ¿qué 3 preguntas le harías para validar esa correspondencia?
  5. Si las coordenadas geoespaciales estuvieran disponibles, como las usarian?

Pueden leer otras columnas del conjunto de AirBnB además de las que están en `interesting_cols`, si les parecen relevantes.

¿Qué cosas no están en los datos que te gustaría tener para predecir mejor el precio de una propiedad?


### Ejercicio 2.1 — Subset de columnas relevantes para predecir Price

**Qué pide la consigna**: subset justificado de columnas, % faltantes, distribución, eliminar outliers, visualizarlos.

#### Cómo elegir las columnas

El objetivo es predecir `Price`. Para cada columna, te preguntás:
1. **¿Tiene info predictiva sobre Price?** Si no correlaciona con el target ni semánticamente ni estadísticamente, no aporta.
2. **¿Es factible procesarla?** Una columna con 80% faltantes o con 13K valores únicos genera más problemas que valor.
3. **¿Es redundante con otra?** Si ya tenés `Rooms`, ¿necesitás también `Bedroom2` (r=0.94)?

#### Decisiones para el subset de TP2

| Columna | Mantener | Justificación |
|---------|----------|---------------|
| `Price` | ✓ | Target. |
| `Rooms` | ✓ | Correlación 0.50 con Price. Discreto, sin faltantes. |
| `Type` | ✓ | h/t/u tienen patrones de precio distintos. Solo 3 categorías. |
| `Distance` | ✓ | Distancia al CBD: capturador clásico de precio inmobiliario. |
| `Bathroom`, `Car`, `Landsize` | ✓ | Tamaño/comodidad → precio. |
| `BuildingArea` | ✓ | r ~0.45 con Price (cuando hay dato). 47% NaN — imputable. |
| `YearBuilt` | ✓ | Año correlaciona con valor (40% NaN, imputable). |
| `Lattitude`, `Longtitude` | ✓ | Geo continuo, útil para clustering y modelos espaciales. |
| `Postcode` | ✓ | Clave para merge AirBnB. |
| `Suburb`, `Regionname` | ✓ | Localización categórica jerárquica. |
| `CouncilArea` | ✓ | Localización gubernamental. 10% NaN tratable. |
| `Propertycount` | ✓ | Densidad del suburbio. |
| `Method` | ✓ | Tipo de venta (S/SP/PI/...). Puede informar sobre apuro del vendedor. |
| **`SellerG`** | ✗ | Agente inmobiliario. 268 categorías. **No es feature de la propiedad sino del proceso**. Si lo usás para predecir Price, es leakage: agentes "premium" venden propiedades premium. Mejor descartar. |
| **`Address`** | ✗ | 13.378 únicos. Identificador, no aporta a OHE/PCA. La info geográfica ya está en otras columnas. |
| **`Bedroom2`** | ✗ | r=0.94 con Rooms (redundante). |
| **`Date`** | △ | Fecha de venta. Probamos en TP1 que tiene baja correlación con Price (\|r\| < 0.05). Para TP2 la dejamos para no perder info temporal por si hace falta. |

**Total**: 15 columnas (de 21 originales).

#### Por qué SellerG es leakage (concepto importante)

Imaginate que sumás SellerG al modelo y descubrís que "agente Jellis Craig" predice precios altos. ¿Causalidad o correlación? Jellis Craig NO causa que la casa valga más — Jellis Craig se ESPECIALIZA en casas caras (hay un sesgo de selección en quién va a cada agente). Si usás esa info para predecir, tu modelo "funciona" en datos viejos pero **no generaliza** a nuevas propiedades (donde no sabés quién va a ser el agente).

Este es uno de los errores más caros en ML: usar features que **no estarían disponibles al momento de la predicción real** o que **contienen información del target indirectamente**.

In [ ]:
# Subset de columnas relevantes
COLS_RELEVANTES = [
    'Price',                                                   # target
    'Rooms', 'Type', 'Distance',                              # tamaño + tipo + ubicación
    'Bathroom', 'Car', 'Landsize',                            # comodidades
    'BuildingArea', 'YearBuilt',                              # imputables
    'Lattitude', 'Longtitude',                                # geo continuo
    'Postcode', 'Suburb', 'Regionname', 'CouncilArea',        # geo categórico
    'Propertycount', 'Method',                                # contexto
    # Date la dejamos por si la queremos en exploración temporal
    'Date',
]

melb_subset = melb_df[COLS_RELEVANTES].copy()
print(f"Subset shape: {melb_subset.shape}  (de {melb_df.shape})")
print(f"\nColumnas descartadas: {set(melb_df.columns) - set(COLS_RELEVANTES)}")


#### Análisis de faltantes en el subset

¿Qué porcentaje de filas tiene AL MENOS UN valor faltante?

In [ ]:
print("=== % FALTANTES POR COLUMNA ===\n")
faltantes = melb_subset.isnull().sum()
pct = (faltantes / len(melb_subset) * 100).round(2)
df_falt = pandas.DataFrame({'faltantes': faltantes, '%': pct}).sort_values('faltantes', ascending=False)
print(df_falt[df_falt['faltantes'] > 0])

# Filas con AL MENOS UN faltante
filas_con_nan = melb_subset.isnull().any(axis=1).sum()
pct_filas = filas_con_nan / len(melb_subset) * 100
print(f"\n=== FILAS CON AL MENOS UN FALTANTE ===")
print(f"{filas_con_nan} filas ({pct_filas:.1f}%)")
print(f"\nObservación: si dropeáramos TODAS las filas con cualquier NaN, perderíamos {pct_filas:.0f}% del dataset.")
print(f"Por eso preferimos imputar (TP1) o tratar el NaN como categoría (CouncilArea).")


#### Distribución de las columnas numéricas

`describe()` nos da min/max/cuartiles/media/std. Lo crítico para outliers: el máximo razonable vs el máximo observado, y la cola larga (std mucho mayor a la diferencia P75-P25 indica cola).

In [ ]:
numericas = melb_subset.select_dtypes(include='number').columns.tolist()
print(f"Numéricas ({len(numericas)}): {numericas}\n")
print("=== DISTRIBUCIÓN ===")
print(melb_subset[numericas].describe().round(2))

# Histograma de Price y otras columnas clave
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flatten(), ['Price', 'Rooms', 'Distance', 'Landsize', 'BuildingArea', 'YearBuilt']):
    melb_subset[col].dropna().hist(bins=50, ax=ax, color='steelblue', alpha=0.7)
    ax.set_title(col)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


#### Detección y eliminación de outliers

Cuáles columnas tienen outliers evidentes:

- **Price**: cola larga (max 9M, P75=1.33M). Outliers altos esperables (casas de lujo).
- **Landsize**: extremo (max 433.014 m² = un campo de 43 hectáreas). Casi seguro errores de carga o propiedades industriales/rurales.
- **BuildingArea**: max 44.515 m² = imposible para una casa (un estadio de fútbol mide ~7.000 m²).
- **Distance**: max 48 km. Razonable, propiedades suburbanas.
- **Bathroom**, **Car**: max 8 y 10 respectivamente. Plausibles para mansiones.

#### Estrategia: IQR para outliers + criterio de dominio

Combinamos las **3 técnicas** del apunte `07-exploracion-eda.md`:
1. **Boxplot** para identificar visualmente.
2. **IQR** como criterio cuantitativo (Q1 - 1.5·IQR, Q3 + 1.5·IQR).
3. **Reglas de dominio**: BuildingArea > 1000 m² casi seguro es error; Landsize > 50.000 m² no es una casa típica.

#### Decisión: qué dropear

- **Landsize > 5.000 m²**: terreno de 50×100 m, ya es grande. Más que eso es campo, no casa.
- **BuildingArea > 1.000 m²**: construcción de 30×30 m, ya es enorme. Más es error.
- **Price > Q3 + 3·IQR**: cola extrema (en vez del 1.5 estándar, usamos 3 para conservar mansiones legítimas pero filtrar errores).
- **NO dropeamos por Bathroom, Car, Rooms**: los rangos son plausibles.

In [ ]:
import numpy

# Identificar outliers por las 3 técnicas combinadas
def detectar_outliers_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (serie < q1 - k*iqr) | (serie > q3 + k*iqr)

# Boxplots para visualizar
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, ['Price', 'Landsize', 'BuildingArea']):
    melb_subset.boxplot(column=col, ax=ax)
    ax.set_title(f'{col} (boxplot)')
plt.tight_layout()
plt.show()

# Reglas de dominio
print("\n=== OUTLIERS A ELIMINAR (criterio de dominio) ===")
mask_landsize  = melb_subset['Landsize'] > 5000
mask_building  = melb_subset['BuildingArea'] > 1000  # NaN no cuenta como > 1000
mask_price     = melb_subset['Price'] > melb_subset['Price'].quantile(0.75) + \
                 3 * (melb_subset['Price'].quantile(0.75) - melb_subset['Price'].quantile(0.25))

print(f"Landsize > 5000 m²:   {mask_landsize.sum()} filas")
print(f"BuildingArea > 1000:  {mask_building.sum()} filas")
print(f"Price extremo (3·IQR): {mask_price.sum()} filas")

mask_outliers = mask_landsize | mask_building | mask_price
print(f"\nTotal a eliminar (cualquier criterio): {mask_outliers.sum()} filas ({mask_outliers.mean()*100:.2f}%)")


#### Visualización de los outliers que vamos a eliminar

Antes de borrar, mostremos visualmente CUÁLES son. La consigna lo pide explícitamente: *"Mostrar visualmente los valores extremos que eliminás"*.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Scatter Price vs BuildingArea, coloreando outliers
sc = axes[0]
no_outlier = melb_subset[~mask_outliers]
yes_outlier = melb_subset[mask_outliers]
sc.scatter(no_outlier['BuildingArea'], no_outlier['Price'], alpha=0.3, s=8, label='Normal', color='steelblue')
sc.scatter(yes_outlier['BuildingArea'], yes_outlier['Price'], alpha=0.8, s=30, label='Outlier', color='red')
sc.set_xlabel('BuildingArea')
sc.set_ylabel('Price')
sc.set_title('Price vs BuildingArea (outliers en rojo)')
sc.legend()

# Scatter Price vs Landsize
sc = axes[1]
sc.scatter(no_outlier['Landsize'], no_outlier['Price'], alpha=0.3, s=8, label='Normal', color='steelblue')
sc.scatter(yes_outlier['Landsize'], yes_outlier['Price'], alpha=0.8, s=30, label='Outlier', color='red')
sc.set_xlabel('Landsize')
sc.set_ylabel('Price')
sc.set_title('Price vs Landsize')
sc.legend()

# Histograma de Price con outliers marcados
ax = axes[2]
ax.hist(no_outlier['Price'].dropna(), bins=60, alpha=0.6, label='Conservados', color='steelblue')
ax.hist(yes_outlier['Price'].dropna(), bins=20, alpha=0.8, label='Eliminados', color='red')
ax.set_xlabel('Price')
ax.set_title('Distribución de Price')
ax.legend()
ax.set_yscale('log')

plt.tight_layout()
plt.show()

# Aplicar el filtro
melb_clean = melb_subset[~mask_outliers].copy()
print(f"\n=== POST-ELIMINACIÓN ===")
print(f"Filas antes:    {len(melb_subset)}")
print(f"Filas después:  {len(melb_clean)}")
print(f"Eliminadas:     {len(melb_subset) - len(melb_clean)} ({(1 - len(melb_clean)/len(melb_subset))*100:.2f}%)")


### Ejercicio 2.2 — Enriquecimiento con AirBnB por zipcode

**Qué pide la consigna**:
1. Elegir variables y combinaciones. ¿Por qué mediana y no media?
2. Unir por zipcode con mínimo de registros (a elección).
3. Gráfico zipcode vs airbnb_price_median.
4. Investigar 2 variables alternativas de join + 3 preguntas a experto inmobiliario.
5. Si tuviéramos coordenadas, ¿cómo las usaríamos?

#### Por qué mediana y no media (responde la consigna)

| Métrica | Para qué | Trampa |
|---------|----------|--------|
| **Mean** | Promedio aritmético | Inflada por outliers (un lujo a $1500/noche tira la media) |
| **Median** | Valor del medio | Robusta a outliers — describe el precio "típico" |

En AirBnB hay cola larga: un departamento de lujo en CBD a $1500/noche convive con un cuarto compartido a $30. La mediana ignora eso; la media se rompe. **Para describir el precio típico de un barrio, la mediana es honesta.**

Esto es exactamente el caso del **breakdown point** del apunte `07-limpieza-y-calidad-de-datos.md` en AVD/v2: la mediana tiene breakdown 50%, la media 0%.

#### Mínimo de registros

Ya pre-filtramos `airbnb_count >= 5` en el setup. Por eso `airbnb_by_zip` solo trae zipcodes con datos significativos. **5 registros es un piso bajo pero suficiente para que la mediana sea representativa**; en producción podría querer 30+ para análisis serios.

In [ ]:
# Merge melb_clean (sin outliers) con airbnb_by_zip
# Cuidar: Postcode en melb es float (por NaN históricos), zipcode en airbnb es int
melb_clean['Postcode_int'] = melb_clean['Postcode'].astype(int)

melb_enriquecido = melb_clean.merge(
    airbnb_by_zip,
    how='left',
    left_on='Postcode_int',
    right_on='zipcode',
    validate='many_to_one',   # garantiza que zipcode es único en airbnb_by_zip
)

print(f"melb_clean shape:         {melb_clean.shape}")
print(f"melb_enriquecido shape:   {melb_enriquecido.shape}")
print(f"\nCobertura del merge:")
print(f"  Filas con AirBnB info:  {melb_enriquecido['airbnb_price_median'].notna().sum()} ({melb_enriquecido['airbnb_price_median'].notna().mean()*100:.1f}%)")
print(f"  Filas sin AirBnB:       {melb_enriquecido['airbnb_price_median'].isna().sum()}")

# Validación post-merge (mismo patrón que en SQL Ej 1.6)
assert len(melb_enriquecido) == len(melb_clean), "El merge cambió filas!"
print(f"\n[OK] Validación: filas se mantienen ({len(melb_enriquecido)})")


#### Gráfico zipcode vs airbnb_price_median

Lo que vamos a observar:
- **Zipcodes con price_median alto**: zonas turísticas/centrales (CBD, costa).
- **Zipcodes con price_median bajo**: suburbios residenciales.
- **Dispersión**: cuánto varía el precio de AirBnB entre zipcodes (proxy de heterogeneidad del mercado).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top 20 zipcodes ordenados por price_median
top20 = airbnb_by_zip.nlargest(20, 'airbnb_price_median')
ax = axes[0]
ax.bar(top20['zipcode'].astype(str), top20['airbnb_price_median'], color='steelblue')
ax.set_xlabel('Zipcode')
ax.set_ylabel('Precio mediano AirBnB (AUD/noche)')
ax.set_title('Top 20 zipcodes por precio mediano AirBnB')
ax.tick_params(axis='x', rotation=45)
ax.grid(alpha=0.3)

# Distribución completa (todos los zipcodes)
ax = axes[1]
zips_sorted = airbnb_by_zip.sort_values('airbnb_price_median')
ax.plot(range(len(zips_sorted)), zips_sorted['airbnb_price_median'], 'o-', alpha=0.5, color='steelblue')
ax.set_xlabel('Zipcode (ordenado)')
ax.set_ylabel('Precio mediano AirBnB')
ax.set_title(f'Distribución completa de precios medianos por zipcode (n={len(zips_sorted)})')
ax.axhline(zips_sorted['airbnb_price_median'].median(), color='red', linestyle='--', label='Mediana global')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Stats descriptivas
print("\n=== DESCRIPTIVA airbnb_price_median ===")
print(airbnb_by_zip['airbnb_price_median'].describe().round(2))


#### Variables alternativas para hacer el join (con 3 preguntas a experto inmobiliario)

**La consigna pregunta** por al menos 2 variables alternativas a `zipcode` para hacer el merge entre `melb_df` y `airbnb_raw`. Vamos a analizar 2 propuestas concretas con sus trade-offs.

#### Alternativa 1: `Suburb` (en melb) ↔ `neighbourhood_cleansed` (en airbnb)

| Aspecto | Análisis |
|---------|----------|
| ¿Disponible en ambos? | Sí: `Suburb` en melb (314 únicos), `neighbourhood_cleansed` en airbnb. |
| ¿Misma granularidad? | Probable. Ambos son barrios. |
| ¿Mismo nombre exactamente? | **Improbable**. "South Yarra" vs "South-Yarra", "Melbourne CBD" vs "CBD", etc. |
| **Trampa** | Spelling, capitalización, separadores. Requiere normalización fuzzy. |
| ¿Cuándo es mejor que zipcode? | Cuando un zipcode cubre MÚLTIPLES barrios distintos (los códigos postales no respetan barrios). |

**3 preguntas al experto inmobiliario antes de usar Suburb como join key**:

1. *"En Melbourne, ¿los nombres de barrios oficiales del consejo municipal coinciden con los que la gente usa en clasificados? ¿Cómo deberíamos mapear barrios populares como 'St Kilda West' que no son oficiales?"*
2. *"¿Hay barrios fronterizos que pertenezcan a 2 suburbios distintos según quién pregunte (e.g., una propiedad en la frontera Northcote/Thornbury)? ¿Cómo se resuelve en la práctica?"*
3. *"¿Cuándo un nombre de barrio cambia formalmente en el catastro? ¿Tenés un mapeo histórico para fusionar datasets de distintos años?"*

#### Alternativa 2: `(Lattitude, Longitude)` (en melb) ↔ `(latitude, longitude)` (en airbnb)

| Aspecto | Análisis |
|---------|----------|
| ¿Disponible en ambos? | Sí, en ambos datasets. |
| ¿Mismo formato? | Sí — decimal latitude/longitude. |
| ¿Match exacto? | **Imposible**: cada propiedad tiene su lat/lon único, no hay un valor para "joinear" directamente. |
| **Solución** | Join geoespacial: para cada propiedad de melb, encontrar las K más cercanas en airbnb_raw y agregar (mediana de Price, etc.). |
| **Costo** | O(N·M) si lo hacés con bucle naive. Mejorable con KD-tree o ball-tree (O(N·log M)). |
| ¿Cuándo es mejor? | Cuando los barrios/zipcodes son granularidad muy gruesa. Permite "AirBnB en un radio de 500m de la propiedad". |

**3 preguntas al experto inmobiliario antes de usar lat/lon como join**:

1. *"¿Qué radio define una vecindad relevante para un comprador? ¿200m? ¿500m? ¿1km? ¿Eso cambia entre CBD y suburbios?"*
2. *"En zonas con barreras físicas (río, autopista, vías), ¿la distancia en línea recta sigue siendo un buen proxy de 'cercanía'?"*
3. *"¿Hay zonas donde las coordenadas reportadas no son precisas (terrenos rurales, edificios en construcción)? ¿Cómo lo detectás?"*

#### Si tuviéramos coordenadas geoespaciales (responde el ítem 5 de la consigna)

Las usaríamos para:

1. **Join por radio**: para cada property, agregar la mediana de price de los N AirBnB dentro de un radio R (con `scipy.spatial.cKDTree` o `geopandas.sjoin_nearest`).
2. **Feature engineering**: distancia al CBD, distancia al supermercado más cercano, densidad de AirBnB en X km — todas son señales fuertes para el modelo.
3. **Clustering**: K-means o DBSCAN en (lat, lon) para identificar "zonas naturales" más finas que los suburbios.
4. **Validación de zipcode**: cazar errores donde el zipcode reportado no corresponde a la lat/lon real (un dato sucio que tenés en un dataset y otro).
5. **Atributos del entorno**: con APIs externas (Google Places, OSM), derivar features tipo "número de cafés en 1km" o "distancia al parque más cercano".

### Criterios de evaluación
Se evaluará principalmente:
- claridad del código,
- justificación de las decisiones de curación,
- coherencia entre el análisis realizado y las conclusiones,
- presencia de validaciones después de operaciones críticas como merges o cargas a base.

No se espera una única solución correcta, pero sí que las decisiones estén justificadas y sean consistentes con los datos.


## Ejercicio 3:

Crear y guardar un nuevo conjunto de datos con todas las transformaciones realizadas anteriormente.

### Ejercicio 3 — Persistencia del dataset final

**Qué pide**: guardar el dataset con todas las transformaciones.

#### Por qué este paso es crítico (aunque parezca trivial)

El dataset crudo de Melbourne tiene 21 columnas. El nuestro tiene ~22 columnas (+5 de AirBnB, -1 de SellerG, -1 de Address, -1 de Bedroom2, +Postcode_int) y ~13K filas (después de filtrar outliers). Si un compañero del próximo trabajo quiere usar tu output como input, **el CSV es el contrato**.

#### Decisiones de persistencia

| Decisión | Por qué |
|----------|---------|
| Archivo CSV (no Parquet) | Estándar académico, todos lo abren |
| `index=False` | El índice de pandas no aporta nada al modelo |
| Verificación post-guardado | Igual que TP1 — releer y validar |

#### Sobre Parquet (extensión)

Si quisieras formato binario eficiente: `df.to_parquet('output.parquet')`. Ventajas:
- 5-10x más chico que CSV.
- Conserva los dtypes (CSV pierde int vs float, datetime, etc.).
- Lectura mucho más rápida.

Costo: necesitás `pyarrow` o `fastparquet`. Para este TP nos quedamos con CSV.

In [ ]:
# Eliminar la columna auxiliar Postcode_int (la creamos para el merge)
output_df = melb_enriquecido.drop(columns=['Postcode_int', 'zipcode']).copy()

OUTPUT_PATH = 'melb_data_enriquecido.csv'
output_df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset guardado en: {OUTPUT_PATH}")
print(f"Shape: {output_df.shape}")

# Verificación post-guardado (mismo patrón que TP1)
df_verif = pandas.read_csv(OUTPUT_PATH)

assert df_verif.shape == output_df.shape, "Shape no coincide tras releer!"
print(f"\n[OK] Shape verificada: {df_verif.shape}")
print(f"[OK] Columnas verificadas: {len(df_verif.columns)}")
print(f"\nResumen del archivo final:")
print(f"  Filas:     {len(df_verif)}")
print(f"  Columnas:  {df_verif.shape[1]}")
print(f"  Con AirBnB enriquecimiento: {df_verif['airbnb_price_median'].notna().sum()} filas")
print(f"  Sin AirBnB enriquecimiento: {df_verif['airbnb_price_median'].isna().sum()} filas")

import os
size_mb = os.path.getsize(OUTPUT_PATH) / 1024 / 1024
print(f"  Tamaño:    {size_mb:.2f} MB")


## Ejercicios opcionales:

El notebook `02.2 ETLs-DAGs.ipynb` tiene un esqueleto de referencia para guiarse.

1. Armar un script en python (archivo .py) [ETL](https://towardsdatascience.com/what-to-log-from-python-etl-pipelines-9e0cfe29950e) que corra los pasos de extraccion, transformacion y carga, armando una funcion para cada etapa del proceso y luego un main que corra todos los pasos requeridos.

2. Armar un DAG en Apache Airflow que corra el ETL. (https://airflow.apache.org/docs/apache-airflow/stable/tutorial.html)

3. Bonus: embeddings y búsqueda semántica con descripciones de AirBnB.
   - Usar `sentence-transformers` para codificar descripciones textuales de propiedades.
   - Tomar un subconjunto chico de descripciones, calcular embeddings y encontrar el par más similar con similitud coseno.
   - Reflexionar: ¿por qué este resultado no se puede lograr con `LIKE '%keyword%'` en SQL? ¿Qué pasa si dos propiedades son similares pero usan palabras distintas? ¿Qué representan los 384 números del embedding?

4. Bonus: curación asistida por IA sobre `CouncilArea`.
   - Tomar los valores únicos de `CouncilArea` y pedirle a un modelo de lenguaje que sugiera posibles inconsistencias, duplicados por capitalización o spelling y agrupaciones razonables.
   - Proponer una estandarización preliminar y luego validar manualmente si el mapeo tiene sentido.
   - Reflexionar: ¿cuándo confiarías en este resultado sin revisarlo? ¿Qué pasa si el modelo inventa un mapeo incorrecto? ¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?


## Ejercicio 4 — Opcionales

### 4.1 — Script ETL en archivo `.py`

**Qué pide**: armar un script Python (`etl.py`) con funciones para extracción, transformación y carga, y un `main` que ejecute todo.

#### Por qué tener un .py separado del notebook

El notebook está bueno para **exploración**. El `.py` está bueno para **ejecución**:
- Versionable con git (los notebooks tienen mucho output que ensucia los diffs).
- Importable desde otros scripts (`from etl import extract, transform, load`).
- Ejecutable por Airflow, cron, CI, etc.
- Testable con pytest.

Es el patrón **Bronze → Silver → Gold** del apunte `09-etl-y-dags.md`, donde cada función representa una capa.

#### Estructura del ETL

```
etl.py
├── connect_db()         # crea engine SQLAlchemy
├── extract()            # baja los CSVs de FAMAF
├── transform()          # limpia, agrega, valida
├── load()               # ingesta a SQLite + guarda CSV final
└── main()               # orchestra: extract -> transform -> load
```

Buenas prácticas que metemos:
- **Logging** con niveles (INFO, ERROR), NO `print`.
- **Funciones puras**: cada paso recibe inputs y devuelve outputs, sin estado global.
- **Path absolutos vía Path**: nada de strings hardcoded.
- **Type hints**: documentan qué espera y devuelve cada función.

In [ ]:
# Generar etl.py desde el notebook
from pathlib import Path

etl_code = '''"""
ETL para el TP2 de EyCD — DiploDatos 2026.

Extrae datos de Melbourne Housing + AirBnB Melbourne desde FAMAF,
limpia, agrega y carga el resultado en SQLite + CSV.

Uso:
    python etl.py
"""
import logging
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger(__name__)

# Constantes
URL_MELB = "https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv"
URL_AIRBNB = "https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/cleansed_listings_dec18.csv"
DB_PATH = Path("melb_etl.sqlite3")
CSV_OUTPUT = Path("melb_data_etl_output.csv")
MIN_REGISTROS_AIRBNB = 5


def connect_db(db_path: Path = DB_PATH) -> Engine:
    """Crea engine SQLAlchemy para SQLite."""
    log.info(f"Conectando a DB: {db_path}")
    return create_engine(f"sqlite:///{db_path}", echo=False)


def extract() -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Descarga los dos datasets de FAMAF."""
    log.info("Extracción: bajando melb_data.csv")
    melb = pd.read_csv(URL_MELB)
    log.info(f"  -> melb shape: {melb.shape}")

    log.info("Extracción: bajando cleansed_listings_dec18.csv")
    airbnb = pd.read_csv(URL_AIRBNB)
    log.info(f"  -> airbnb shape: {airbnb.shape}")

    return melb, airbnb


def _limpiar_precio(serie: pd.Series) -> pd.Series:
    """Convierte string '$1,234.56' a float 1234.56."""
    return (
        serie.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace("nan", np.nan)
        .astype(float)
    )


def transform(melb: pd.DataFrame, airbnb: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Limpia y agrega los dos datasets."""
    log.info("Transformación: limpiando precios AirBnB")
    airbnb["price"] = _limpiar_precio(airbnb["price"])
    airbnb["weekly_price"] = _limpiar_precio(airbnb["weekly_price"])
    airbnb["monthly_price"] = _limpiar_precio(airbnb["monthly_price"])
    airbnb["zipcode"] = pd.to_numeric(airbnb["zipcode"], errors="coerce")
    airbnb = airbnb.dropna(subset=["zipcode"])
    airbnb["zipcode"] = airbnb["zipcode"].astype(int)

    log.info("Transformación: agregando AirBnB por zipcode")
    airbnb_by_zip = (
        airbnb.groupby("zipcode")
        .agg(
            airbnb_price_median=("price", "median"),
            airbnb_price_mean=("price", "mean"),
            airbnb_count=("price", "count"),
        )
        .reset_index()
    )
    airbnb_by_zip = airbnb_by_zip[airbnb_by_zip["airbnb_count"] >= MIN_REGISTROS_AIRBNB]
    log.info(f"  -> zipcodes válidos: {len(airbnb_by_zip)}")

    log.info("Transformación: limpiando outliers de melb")
    mask_land = melb["Landsize"] > 5000
    mask_build = melb["BuildingArea"] > 1000
    mask_price = melb["Price"] > melb["Price"].quantile(0.75) + 3 * (
        melb["Price"].quantile(0.75) - melb["Price"].quantile(0.25)
    )
    n_outliers = (mask_land | mask_build | mask_price).sum()
    log.info(f"  -> outliers eliminados: {n_outliers}")
    melb = melb[~(mask_land | mask_build | mask_price)].copy()

    log.info("Transformación: convirtiendo Date a datetime")
    melb["Date"] = pd.to_datetime(melb["Date"], format="%d/%m/%Y")
    melb["Postcode_int"] = melb["Postcode"].astype(int)

    return melb, airbnb_by_zip


def load(melb: pd.DataFrame, airbnb_by_zip: pd.DataFrame, engine: Engine) -> pd.DataFrame:
    """Carga a SQLite + merge final + guarda CSV."""
    log.info("Load: ingestando tablas a SQLite")
    melb.to_sql("properties", con=engine, if_exists="replace", index=False)
    airbnb_by_zip.to_sql("airbnb", con=engine, if_exists="replace", index=False)
    log.info(f"  -> properties: {len(melb)} filas")
    log.info(f"  -> airbnb: {len(airbnb_by_zip)} filas")

    log.info("Load: merge final + guardar CSV")
    output = melb.merge(
        airbnb_by_zip,
        how="left",
        left_on="Postcode_int",
        right_on="zipcode",
        validate="many_to_one",
    )
    output = output.drop(columns=["Postcode_int", "zipcode"])
    output.to_csv(CSV_OUTPUT, index=False)
    log.info(f"  -> CSV guardado en {CSV_OUTPUT} ({len(output)} filas)")

    # Validación post-load
    df_verif = pd.read_csv(CSV_OUTPUT)
    assert df_verif.shape == output.shape, "Shape no coincide tras releer!"
    log.info(f"  -> [OK] Verificación post-guardado")

    return output


def main():
    """Orquesta el pipeline."""
    log.info("=== ETL TP2 EyCD ===")
    engine = connect_db()
    melb, airbnb = extract()
    melb_clean, airbnb_by_zip = transform(melb, airbnb)
    final = load(melb_clean, airbnb_by_zip, engine)
    log.info(f"=== ETL COMPLETADO: {final.shape} ===")
    return final


if __name__ == "__main__":
    main()
'''

etl_path = Path('etl.py')
etl_path.write_text(etl_code, encoding='utf-8')
print(f"Script ETL guardado en: {etl_path.resolve()}")
print(f"Tamaño: {etl_path.stat().st_size} bytes")
print(f"\nPara ejecutarlo desde terminal:")
print(f"    python {etl_path.name}")


### 4.2 — DAG Apache Airflow

**Qué pide**: armar un DAG que corra el ETL.

#### Por qué un DAG y no solo el script .py

Un script `.py` se corre **manualmente o por cron**. Un DAG de Airflow agrega:
- **Scheduling declarativo**: "correr cada día a las 3 AM".
- **Dependencias entre tareas**: `t1 >> t2 >> t3` define el orden.
- **Reintentos automáticos** si una tarea falla.
- **UI web** para monitorear ejecuciones, ver logs, re-correr fallos.
- **Backfill**: correr el pipeline para fechas históricas.

#### Estructura del DAG

```
extract >> transform >> load
```

Tres tareas secuenciales, cada una llamando a las funciones de `etl.py`. **El DAG NO procesa datos** — solo coordina (recordá la cita de la cátedra en `09-etl-y-dags.md`: *"Airflow no procesa datos, solo coordina las tareas que sí lo hacen"*).

#### Nota: no necesitás Airflow instalado para escribir el DAG

Vamos a generar `dag_tp2_etl.py` que es código válido de Airflow. Para ejecutarlo necesitarías `pip install apache-airflow` (~500MB de deps) y un broker Celery/Redis o el `SequentialExecutor` para dev. Como no es parte del scope del TP, escribimos el código pero **no lo ejecutamos**.

In [ ]:
dag_code = '''"""
DAG de Airflow para el ETL del TP2 EyCD.

Ubicación esperada: dags/dag_tp2_etl.py dentro de AIRFLOW_HOME.

Para correr (asumiendo Airflow instalado):
    airflow dags trigger tp2_etl
"""
from datetime import datetime, timedelta
from pathlib import Path
import sys

# Asegurar que etl.py es importable
ETL_DIR = Path(__file__).parent.parent
sys.path.insert(0, str(ETL_DIR))

from airflow import DAG
from airflow.operators.python import PythonOperator

import etl  # nuestro módulo

default_args = {
    "owner": "diplodatos-2026",
    "depends_on_past": False,
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
}

with DAG(
    dag_id="tp2_etl",
    description="ETL Melbourne Housing + AirBnB",
    default_args=default_args,
    start_date=datetime(2026, 5, 1),
    schedule_interval="@daily",
    catchup=False,
    tags=["diplodatos", "tp2", "eycd"],
) as dag:

    def task_extract(**kwargs):
        melb, airbnb = etl.extract()
        # XCom: pasar shape a la siguiente tarea (no el df entero — es muy grande)
        kwargs["ti"].xcom_push(key="melb_shape", value=list(melb.shape))
        kwargs["ti"].xcom_push(key="airbnb_shape", value=list(airbnb.shape))
        return "OK"

    def task_transform_and_load(**kwargs):
        # En producción usarías un volumen compartido o un staging area.
        # Acá simplificamos volviendo a extraer dentro de la tarea.
        engine = etl.connect_db()
        melb, airbnb = etl.extract()
        melb_clean, airbnb_by_zip = etl.transform(melb, airbnb)
        output = etl.load(melb_clean, airbnb_by_zip, engine)
        return f"Procesadas {len(output)} filas"

    extract_op = PythonOperator(
        task_id="extract",
        python_callable=task_extract,
    )

    transform_load_op = PythonOperator(
        task_id="transform_and_load",
        python_callable=task_transform_and_load,
    )

    extract_op >> transform_load_op
'''

from pathlib import Path
dag_path = Path('dag_tp2_etl.py')
dag_path.write_text(dag_code, encoding='utf-8')
print(f"DAG guardado en: {dag_path.resolve()}")
print(f"Tamaño: {dag_path.stat().st_size} bytes")
print(f"\nNota: NO está ejecutado, solo está escrito.")
print(f"Para usarlo: copiarlo a $AIRFLOW_HOME/dags/ y tener Airflow instalado.")


### 4.3 — Embeddings y búsqueda semántica

**Qué pide**: usar `sentence-transformers` para encodear descripciones de AirBnB, encontrar el par más similar con similitud coseno, y reflexionar.

#### Por qué embeddings y no LIKE

Imaginate dos descripciones:
- A: *"Cozy apartment near tram station with garden view"*
- B: *"Comfortable flat close to public transport, garden views"*

Hacen el mismo punto, **pero no comparten palabras clave exactas**. `LIKE '%cozy%'` no encuentra B; `LIKE '%comfortable%'` no encuentra A. Necesitarías construir un diccionario manual de sinónimos.

Los **embeddings** convierten cada descripción en un vector de N dimensiones (típicamente 384 con `all-MiniLM-L6-v2`) en el que descripciones **semánticamente parecidas** quedan cerca en el espacio vectorial. La similitud coseno mide ese "cerca" como el ángulo entre vectores.

#### Anatomía del modelo

`all-MiniLM-L6-v2`:
- **Salida**: vector de 384 floats.
- **Entrada**: hasta 256 tokens (~200 palabras).
- **Tamaño del modelo**: ~80 MB.
- **Velocidad**: ~5000 frases/seg en CPU moderna.

#### Lo que aprendés con esto

Si un cliente busca "depto luminoso con balcón", el LIKE clásico falla:
- Una descripción dice "*bright loft with terrace*" → el LIKE no la encuentra (no aparece "luminoso" ni "balcón").
- El embedding las pone cerca porque entiende **el concepto**.

Esto es la base de RAG (Retrieval Augmented Generation), búsqueda semántica, recomendadores. La nueva generación de buscadores.

In [ ]:
# Embeddings con sentence-transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Cargar modelo (la primera vez descarga ~80MB)
print("Cargando modelo sentence-transformer...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Modelo cargado. Dimensión del embedding: {model.get_sentence_embedding_dimension()}")

# Tomar un subset chico de descripciones AirBnB no-nulas
descripciones = airbnb_raw['description'].dropna().head(50).tolist()
print(f"\nDescripciones a encodear: {len(descripciones)}")

# Encodear todas
print("Generando embeddings...")
embeddings = model.encode(descripciones, show_progress_bar=False)
print(f"Shape de la matriz de embeddings: {embeddings.shape}")

# Calcular matriz de similitudes coseno entre todas las pares
sim_matrix = cosine_similarity(embeddings)
# Poner la diagonal en -1 para que np.argmax no devuelva (i,i)
import numpy
numpy.fill_diagonal(sim_matrix, -1)

# Encontrar el par más similar
i, j = numpy.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
print(f"\n=== PAR MÁS SIMILAR ===")
print(f"Similitud coseno: {sim_matrix[i, j]:.4f}\n")
print(f"Descripción {i}:")
print(f"  {descripciones[i][:300]}...\n")
print(f"Descripción {j}:")
print(f"  {descripciones[j][:300]}...")


#### Reflexión (responde lo que pide la consigna)

**¿Por qué no se logra con `LIKE '%keyword%'` en SQL?**

LIKE compara strings **carácter por carácter**. No entiende:
- Sinónimos ("apartment" vs "flat")
- Paráfrasis ("near tram station" vs "close to public transport")
- Idiomas (un texto en inglés no encuentra su equivalente en español)
- Acentos y variaciones ortográficas

Embeddings comparan **representaciones semánticas**: si dos textos describen el mismo concepto, sus vectores quedan cerca.

**¿Qué pasa si dos propiedades son similares pero usan palabras distintas?**

LIKE las trata como diferentes; embeddings las acerca. Por ejemplo, "cozy" y "comfortable" tienen vectores con alta similitud (~0.7+) aunque sean palabras distintas. El modelo lo aprendió leyendo billones de textos donde se usan en contextos parecidos.

**¿Qué representan los 384 números del embedding?**

Cada uno es una "dimensión semántica latente" — algo que el modelo aprendió a detectar durante entrenamiento. NO son interpretables individualmente (no hay "la dimensión que mide cuán acogedor"), pero juntos forman una posición en un espacio donde:
- Distancia = semejanza semántica.
- Direcciones = ejes de variación (en modelos famosos como word2vec, hay direcciones que codifican "género", "tiempo verbal", etc., pero solo se descubren a posteriori).

Es **lo opuesto a OHE**: OHE genera 1 dimensión por palabra única (ortogonalidad cruda); embeddings comprimen el significado en 384 floats densos donde la cercanía importa.

### 4.4 — Curación de `CouncilArea` asistida por LLM

**Qué pide**: pedirle a un LLM que detecte inconsistencias en `CouncilArea`, proponer estandarización, reflexionar.

#### Estrategia con y sin API key

Vamos a hacer DOS implementaciones:

1. **Demo CON `anthropic.Client`**: el código que pide la consigna. Si tenés API key, lo corrés. Si no, lo lees y entendés el patrón.
2. **Fallback CON `difflib`**: detecta inconsistencias de spelling sin LLM. Útil para validar el flujo sin gastar tokens.

#### Por qué `difflib` no reemplaza al LLM

`difflib.get_close_matches` solo encuentra strings parecidos por edit distance ("Yarra" vs "Yara"). No entiende:
- Sinónimos administrativos ("Council of X" vs "X Council").
- Conocimiento de dominio ("CBD" pertenece a "City of Melbourne").
- Capitalización contextual.

El LLM SÍ, porque tiene conocimiento general del mundo (sabe que Melbourne tiene un consejo llamado "City of Melbourne", etc.).

In [ ]:
import difflib
from collections import Counter

# === Versión 1: detección de inconsistencias con difflib (sin LLM) ===
council_unique = melb_df['CouncilArea'].dropna().unique().tolist()
print(f"Valores únicos de CouncilArea: {len(council_unique)}\n")
print("Lista completa:")
for c in sorted(council_unique):
    count = (melb_df['CouncilArea'] == c).sum()
    print(f"  {c:30}  (n={count})")

# Buscar pares de valores muy parecidos (posibles duplicados con typo)
print("\n=== POSIBLES DUPLICADOS (difflib similarity > 0.85) ===")
pares_sospechosos = []
for i, a in enumerate(council_unique):
    matches = difflib.get_close_matches(a, council_unique[i+1:], n=3, cutoff=0.85)
    for m in matches:
        pares_sospechosos.append((a, m))
        print(f"  '{a}' <-> '{m}'")

if not pares_sospechosos:
    print("  (ninguno con similitud > 0.85)")


In [ ]:
# === Versión 2: curación con LLM (Claude) — REQUIERE API KEY ===
# Si tenés ANTHROPIC_API_KEY en el environment, este código corre.
# Si no, queda como demo del patrón.

import os
import json

api_key_disponible = bool(os.getenv('ANTHROPIC_API_KEY'))
print(f"ANTHROPIC_API_KEY disponible: {api_key_disponible}")

if api_key_disponible:
    import anthropic
    client = anthropic.Anthropic()

    prompt = f'''Estos son los valores únicos de la columna CouncilArea en un dataset de propiedades de Melbourne:
{council_unique}

Identificá: (1) duplicados con distinta capitalización o spelling,
(2) valores que parecen errores, (3) valores que podrían agruparse.

Respondé en JSON con esta estructura:
{{"estandarizado": {{"valor_original": "valor_correcto"}}, "agrupaciones": [{{"grupo": "nombre", "valores": [...]}}], "observaciones": [...]}}'''

    print("Enviando prompt a Claude...")
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2048,
        messages=[{"role": "user", "content": prompt}]
    )
    response_text = message.content[0].text
    print(f"\nRespuesta de Claude:\n{response_text[:2000]}")
else:
    print("\nDemo: no se ejecutó la llamada al LLM.")
    print("Para correrlo, exportá ANTHROPIC_API_KEY antes de levantar Jupyter:")
    print("    export ANTHROPIC_API_KEY=sk-ant-...")
    print("\nQué haría el LLM con un prompt como este:")
    print("  - Detectaría duplicados ortográficos ('Yarra' vs 'Yara' — si existieran)")
    print("  - Agruparía consejos vecinos por similitud administrativa")
    print("  - Marcaría valores que parecen errores ('Unknown' que sí existe en nuestro dataset)")


#### Reflexión sobre confiar en el LLM (responde la consigna)

**¿Cuándo confiarías en el resultado sin revisarlo?**

NUNCA en producción sin checks. Sí confiaría como **HINT** para:
- Detectar inconsistencias obvias rápido (capitalización, spelling de typos).
- Sugerir agrupaciones cuando hay miles de valores únicos (donde la inspección manual es inviable).
- Generar un MAPPING preliminar que después validás contra una fuente autoritativa (catastro, registro civil, etc.).

**¿Qué pasa si el modelo inventa un mapeo incorrecto?**

Los LLMs **alucinan**: pueden generar mappings con confianza pero falsos ("Boroondara" → "Borondara" si el modelo no conoce el barrio real). Estrategias para mitigarlo:

1. **Two-step validation**: usar el LLM para SUGERIR, validar con una segunda fuente (lista oficial de consejos).
2. **Cross-checking**: pedirle al MISMO LLM 2 veces con prompts levemente distintos. Si las respuestas difieren, hay duda.
3. **Bajar la temperatura**: en `anthropic.Client` agregás `temperature=0.0` para minimizar variabilidad creativa.
4. **Confianza calibrada**: pedir al LLM que devuelva una **confianza** (0-1) por cada mapeo y filtrar solo los > 0.9.

**¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?**

No podés inspeccionar 1000+ mappings a mano. Estrategias:

1. **Sample stratified**: validar 10% al azar + 10% de los mappings con menor confianza reportada.
2. **Reverse lookup**: si el LLM mapeó "X" → "Y", verificar que "Y" es un valor que SÍ existe en una fuente autoritativa.
3. **Cluster validation**: si el LLM agrupa 50 valores bajo "Council of Melbourne", verificar que esos 50 valores tienen lat/lon en la zona geográfica de City of Melbourne.
4. **Monitoreo continuo**: si el ETL se corre periódicamente, **trackear** qué % de valores nuevos requieren mapeo nuevo y revisar drift.